<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# Cree Agentes LLM Interactivos con Herramientas

En este laboratorio, explorará las poderosas capacidades de las llamadas a herramientas en grandes modelos de lenguaje (LLM) para crear agentes de IA avanzados que puedan interactuar dinámicamente con los usuarios. Utilizando el marco LangChain, aprenderá cómo crear un agente interactivo que responda a las consultas de los usuarios seleccionando y ejecutando la función correcta en el momento correcto. Este enfoque práctico le ayudará a comprender cómo se pueden ampliar los LLM con funcionalidad del mundo real, uniendo la comprensión del lenguaje natural con acciones dinámicas basadas en herramientas.


## 1. Objetivos

Después de completar este laboratorio podrás:

 - Inicializar un modelo de chat para interacciones de herramientas
 - Definir y vincular herramientas personalizadas al LLM para ampliar la funcionalidad
 - Utilice diccionarios de mapeo para llamadas a funciones dinámicas
 - Extraer nombres de herramientas y funciones para llamadas de funciones precisas
 - Cree clases de agentes que administren todo el proceso de llamada a herramientas


## 2. Configuración


Para este laboratorio, utilizará las siguientes bibliotecas:

* [`langchain`](https://python.langchain.com/docs/introduction/) es el marco sobre el que construirá el agente.
* [`langchain-openai`](https://pypi.org/project/langchain-openai/) es un paquete asociado de LangChain e integra los LLM de OpenAI al marco.


### *2.1. Instalación de las Bibliotecas Necesarias*

Las siguientes bibliotecas necesarias no están preinstaladas en el su entorno. Debe ejecutar la siguiente celda para instalarlas. Este paso puede tardar varios minutos, tenga paciencia.

```bash
pip install -r requirements.txt
```

### *2.2. Importación de las Bibliotecas Necesarias*

Importa aquí todas las bibliotecas necesarias:

In [2]:
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

Inicialicemos el modelo de lenguaje que potenciará las capacidades de llamada de su herramienta. Este código configura un modelo GPT-4o-mini utilizando el proveedor OpenAI a través de la interfaz de LangChain, que utilizará para procesar consultas y decidir a qué herramientas llamar.


In [3]:
from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-4o-mini", model_provider= "openai")

### *2.3. Aviso Legal Sobre la API*

Este laboratorio utiliza módulos de lenguaje natural (LLM) proporcionados por **IBM watsonx.ai** y **OpenAI**. Este entorno se ha configurado para permitir el uso de LLM sin claves API, de modo que pueda solicitarlos **gratis (con limitaciones)**. Por lo tanto, si desea ejecutar este cuaderno **localmente fuera** del entorno JupyterLab de Skills Network, deberá **configurar sus propias claves API**. Tenga en cuenta que el uso de sus propias claves API implicará cargos personales.

Si ejecuta este laboratorio localmente, deberá configurar sus propias claves API. Este laboratorio utiliza los módulos `ChatOpenAI` y `ChatWatsonx` de `langchain`. Ambas configuraciones se muestran a continuación con sus instrucciones. **Reemplace todas las instancias** de ambos módulos con los módulos completos que se muestran a continuación en todo el laboratorio. **NO** ejecute la celda que se muestra a continuación si no está trabajando localmente, ya que provocará errores.

In [4]:
from langchain_openai import ChatOpenAI
from langchain_ibm import ChatWatsonx

llm_openai = ChatOpenAI(
    model= 'gpt-4o-mini',
    temperature= 0.7,
    max_completion_tokens= 2048
)

# watsonx_llm = ChatWatsonx(
#     model_id="ibm/granite-3-2-8b-instruct",
#     url="https://us-south.ml.cloud.ibm.com",
#     project_id="your project id associated with the API key",
#     api_key="your watsonx.ai api key here",
# )

## 3. Creación de Herramientas Personalizadas con LangChain

### *3.1. Anatomía de una Herramienta*

Vamos a proporcionar los componentes básicos de una herramienta. Consideremos la siguiente herramienta:

```python
@tool
def tool_name(parametro_entrada: tipo_entrada) -> tipo_salida:
    """
    Descripción clara de lo que hace la herramienta.

    Argumentos:
    parametro_entrada (tipo_entrada): Descripción de este parámetro

    Devuelve:
    tipo_salida: Descripción del valor devuelto
    """

    # Implementación de la función.
    resultado = process(parametro_entrada)

    return resultado
```

### *3.2. Componentes Clave*

Utilizarás los siguientes componentes clave:

**Decorador @tool**

- Registra la función en LangChain

- Crea atributos de herramienta (.name, .description, .func)

- Genera un esquema JSON para la validación

- Transforma funciones regulares en herramientas invocables

**Nombre de la función**

- Lo utiliza LLM para seleccionar la herramienta adecuada

- Se utiliza como referencia en cadenas y asignaciones de herramientas

- Aparece en los registros de llamadas de la herramienta para la depuración

- Debe indicar claramente el propósito de la herramienta

**Anotaciones de tipo**

- Permite la validación automática de la entrada

- Crea un esquema para los parámetros

- Permite la serialización adecuada de entradas/salidas

- Ayuda a LLM a comprender los formatos de entrada requeridos

**Cadena de documentación**

- Proporciona contexto a LLM para decidir cuándo usar la herramienta

- Documenta los requisitos de los parámetros

- Explica las salidas y el comportamiento esperados

- Es fundamental para la selección de herramientas por parte de LLM

6. **Implementación**

- Ejecuta la operación real

- Gestiona los errores adecuadamente

- Devuelve resultados con el formato correcto

- Debe ser eficiente y robusto

### *3.3. Definición de una Función de Adición*

Ahora utilice este marco de herramientas para crear una herramienta personalizada que permita al LLM realizar sumas básicas.

In [5]:
@tool
def agregar(a: int, b: int) -> int:
    """
    Agregar a y b.
    
    Args:
        a (int): primer entero a ser agregado
        b (int): segundo entero a ser agregado

    Return:
        int: suma de a y b
    """
    
    return a + b

El decorador envuelve la función `agregar()` en el esquema de herramienta predefinido de LangChain. Obtenga más información sobre cómo definir herramientas LangChain personalizadas [aquí](https://python.langchain.com/docs/how_to/custom_tools/).

### *3.4. Agregar Herramientas al LLM*

Conectemos y vinculemos la función al modelo de chat.

In [6]:
herramientas = [agregar]

llm_con_herramientas = llm.bind_tools(herramientas)

Utilice el método `bind_tools(tools)` para conectar una lista de herramientas al LLM para su uso. De ahora en adelante, cada vez que se invoque la llamada, el modelo (con herramientas) reconocerá y utilizará la herramienta agregar siempre que necesite calcular una suma.

### *3.5. Crear más Herramientas*

Creemos algunas herramientas aritméticas más básicas.

In [7]:
@tool
def restar(a: int, b:int) -> int:
    """Resta b de a."""
    return a - b

@tool
def multiplicar(a: int, b:int) -> int:
    """Multiplica a y b."""
    return a * b

### *3.6. Probando las Funciones*


Configuremos una forma de probar sus herramientas.

In [9]:
map_herramientas = {
    "agregar": agregar, 
    "restar": restar,
    "multiplicar": multiplicar
}

entradas_ = {
    "a": 1,
    "b": 2
}

map_herramientas["agregar"].invoke(entradas_)

3

Usando el método `.invoke(inputs)` integrado de LangChain, puede probar cada herramienta creada con entradas dinámicas. Pruebe cada herramienta con el bloque de código anterior.

### *3.7. Agregar Nuevas Herramientas a LLM*


Agreguemos las tres herramientas al LLM.

In [10]:
herramientas = [agregar, restar, multiplicar]

llm_con_herramientas = llm.bind_tools(herramientas)

Puede utilizar el mismo método para vincular herramientas al LLM, lo que permite más capacidades aritméticas.

## 4. Interactuando con el Modelo

### *4.1. Elaborar la Consulta del Usuario*

Ahora que ha configurado un LLM con integraciones de herramientas básicas, es hora de presentar las consultas de los usuarios.

In [11]:
consulta = "¿Cuanto es 3 + 2?"

historial_chat = [HumanMessage(content= consulta)]

Primero, configure la pregunta (consulta de usuario). Luego, inicialice una matriz `historial_chat` que contendrá toda la conversación entre el usuario y LLM. En este historial de chat, inserta la "consulta" en un contenedor "HumanMessage" que le dice a LangChain y al modelo: "Este mensaje vino del usuario".

### *4.2. Invocar el Modelo*

Ahora ejecutemos el modelo con el contexto (historial de chat) que contiene la consulta del usuario.

In [12]:
respuesta_1 = llm_con_herramientas.invoke(historial_chat)
historial_chat.append(respuesta_1)

print(type(respuesta_1))

<class 'langchain_core.messages.ai.AIMessage'>


Usando el método `invoke(inputs)`, obtienes una respuesta del modelo. Agregas la respuesta al historial de chat. El bloque de código también imprime el tipo de respuesta, que es la clase "AIMessage" de LangChain. Descomente la segunda declaración impresa y lea los campos de la respuesta "AIMessage".

### *4.3. Llamadas a la Herramienta de Análisis*

Ahora que tiene la respuesta del modelo, puede analizar la respuesta para obtener instrucciones de llamada a la herramienta.

In [15]:
tool_calls_1 = respuesta_1.tool_calls

nombre_herramienta_1 = tool_calls_1[0]["name"]
args_herramienta_1 = tool_calls_1[0]["args"]
id_call_herramienta_1 = tool_calls_1[0]["id"]

print(f'nombre de la herramienta:\n {nombre_herramienta_1}')
print(f'argumentos de la herramienta:\n {args_herramienta_1}')
print(f'ID de la llamada a la herramienta:\n {id_call_herramienta_1}')

nombre de la herramienta:
 agregar
argumentos de la herramienta:
 {'a': 3, 'b': 2}
ID de la llamada a la herramienta:
 call_wRFoqNRRxjOnt3dIdGsoAVeL


- Al extraer el "nombre" de la primera llamada se obtiene el nombre de la herramienta a utilizar. 
    - `agregar` en este caso

- Extraer los `args` proporciona las entradas para pasar a la herramienta. 
    - `{a: 3, b: 2}` en este caso

- Al extraer el `id` se obtiene el identificador único para la llamada a la herramienta. 
    - El ID será diferente cada vez, vinculando las llamadas a las herramientas con sus respectivas respuestas. 
    
    - Crucial para diferenciar llamadas a la misma herramienta y llamadas a herramientas paralelas

### *4.4. Invocar la Herramienta*

Dados los detalles de la llamada a la herramienta del LLM, invoque la herramienta correcta con los argumentos correctos.

In [16]:
respuesta_herramienta = map_herramientas[nombre_herramienta_1].invoke(args_herramienta_1)
herramienta_message = ToolMessage(content= respuesta_herramienta, tool_call_id= id_call_herramienta_1)

print(herramienta_message)

content='5' tool_call_id='call_wRFoqNRRxjOnt3dIdGsoAVeL'


Utilice `map_herramienta`, pasando el nombre de la herramienta y los parámetros para obtener una respuesta. Luego, incluya esa respuesta en un objeto `ToolMessage` de LangChain junto con el ID de llamada de la herramienta. Esta acción permite que el modelo y LangChain procesen mejor las respuestas de las herramientas y la conversación general entre el usuario, el modelo y la herramienta. No dude en descomentar la declaración impresa para ver cómo se ve "herramienta_message".

In [17]:
historial_chat.append(herramienta_message)

A continuación, agregue "herramienta_message" al "historial_chat" para que el modelo conserve el contexto y vea la conversación anterior para una mejor experiencia de conversación. Ahora el historial de chat contiene un `HumanMessage` (consulta inicial del usuario), un `AIMessage` (la respuesta del modelo) y un `ToolMessage` (el resultado de la herramienta).

In [18]:
historial_chat

[HumanMessage(content='¿Cuanto es 3 + 2?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 137, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_3b264ad91d', 'id': 'chatcmpl-DdtgOelxkJtGqJ2IPl8jBUiI0Byyi', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e10f8-df56-7dc1-8af8-5da30e324314-0', tool_calls=[{'name': 'agregar', 'args': {'a': 3, 'b': 2}, 'id': 'call_wRFoqNRRxjOnt3dIdGsoAVeL', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 137, 'output_tokens': 18, 'total_tokens': 155, 'input_token_details': {'audio': 0, 

### *4.5. Generar una Respuesta Final a Partir del Historial de Chat*

Como paso final, pase todo el `historial_chat` al LLM una vez más para obtener una respuesta final.

In [19]:
respuesta_final = llm_con_herramientas.invoke(historial_chat)

print(type(respuesta_final))
print(respuesta_final.content)

<class 'langchain_core.messages.ai.AIMessage'>
3 + 2 es igual a 5.


La impresión de `answer.content` (campo de contenido del objeto `AIMessage`) proporciona el resultado final del LLM para la consulta del usuario. Ha finalizado una interacción completa entre el usuario y el modelo.

## 5. Construyendo un Agente

Puede agrupar toda la funcionalidad anterior en una clase de Agente unificada.

In [21]:
class AgenteLlamadaHerramienta:
    def __init__(self, llm):
        self.llm_con_herramientas = llm.bind_tools(herramientas)
        self.map_herramientas = map_herramientas

    def run(self, consulta: str) -> str:
        # Paso 1: Iniciar historial de chat con consulta del usuario.
        historial_chat = [HumanMessage(content= consulta)]

        # Paso 2: LLM elge herramienta y argumentos.
        respuesta = self.llm_con_herramientas.invoke(historial_chat)

        if not respuesta.tool_calls:
            return respuesta.content # Respuesta directa, no se llamó a ninguna herramienta.
        
        # Paso 3: Extraer información de la herramienta llamada.
        llamar_herramienta = respuesta.tool_calls[0]
        nombre_herramienta = llamar_herramienta["name"]
        args_herramienta = llamar_herramienta["args"]
        call_id_herramienta = llamar_herramienta["id"]

        # Paso 4: Llamar a la herramienta usando el map_herramientas.
        resultado_herramienta = self.map_herramientas[nombre_herramienta].invoke(args_herramienta)

        # Paso 5: Enviar resultado de vuelta al LLM.
        herramienta_message = ToolMessage(content= str(resultado_herramienta), tool_call_id= call_id_herramienta)
        historial_chat.extend([respuesta, herramienta_message])

        # Paso 6: Resultado final del LLM.
        respuesta_final = self.llm_con_herramientas.invoke(historial_chat)
        
        return respuesta_final.content

Este agente realiza exactamente el mismo proceso que el anterior, excepto que la interacción con el manejo del modelo está contenida dentro del método `run()`.

In [22]:
mi_agente = AgenteLlamadaHerramienta(llm)

print(mi_agente.run("uno mas 2"))

print(mi_agente.run("uno - 2"))

print(mi_agente.run("3 por 2"))

La suma de 1 más 2 es igual a 3.
La respuesta de \( 1 - 2 \) es \(-1\).
3 por 2 es igual a 6.


A continuación se muestran tres ejemplos del agente en uso. Estos agentes son dinámicos y pueden manejar muchos tipos y formatos diferentes de datos como entrada. Esta capacidad es un beneficio importante de los agentes de IA, ya que no es necesario normalizar ni formatear los datos de entrada de una manera determinada.

## 6. Conclusión

Ya ha completado esta breve introducción a la creación de agentes de llamadas de herramientas interactivas. Ahora puedes:
- Estructurar las interacciones de los usuarios y configurar modelos de chat para conversaciones en tiempo real y contextualizadas.
- Extraiga nombres y argumentos de herramientas para que coincidan con precisión con la intención del usuario.
- Analizar instrucciones de herramientas complejas, incluido el manejo de múltiples llamadas de herramientas.
- Cree y refine una clase de agente para automatizar todo el proceso de llamada de herramientas.
- Demostrar cómo estos componentes trabajan juntos para transformar los LLM de respondedores pasivos a agentes inteligentes.

## 7. Ejercicios


### *Ejercicio 1: Crear una Nueva Herramienta*


Utilice el formato de herramienta de ejemplo proporcionado en el cuaderno para crear una nueva herramienta llamada "calcular_propina" que toma una "factura_total y un porcentaje de propina" y devuelve el monto de la propina. </br>

Defina e invoque la herramienta con entradas de muestra como `factura_total= 120`, `porcentaje_propina= 15`. </br>

Cree un `map_herramientas` con la herramienta `calcular_propina`.

In [24]:
@tool
def calcular_propina(factura_total: float, porcentaje_propina: float) -> float:
    """Calcula el monto de la propina dado el total de la factura y el porcentaje de propina."""
    return factura_total * (porcentaje_propina / 100)

entradas = {
    "factura_total": 120,
    "porcentaje_propina": 15
}

map_herramientas_2 = {
    "calcular_propina": calcular_propina
}

<details>
    <summary>Click here for the solution</summary>

```python
@tool
def calculate_tip(total_bill: int, tip_percent: int) -> int:
    """Calculate tip"""
    return total_bill * tip_percent * 0.01

inputs = {
    "total_bill": 120,
    "tip_percent": 15
}
calculate_tip.invoke(inputs)


tool_map = {
    "calculate_tip": calculate_tip
}
```

</details>


### *Ejercicio 2: Llm a la Herramienta con un LLM*

Simule una consulta de usuario como "¿Cuánto debo dar de propina por $60 al 20%?". </br>
Vincule la herramienta al `llm` predefinido y solicite al LLM la consulta anterior. Luego, analice la respuesta de LLM para conocer los detalles de la llamada a la herramienta e invoque la herramienta en consecuencia. Finalmente, tome todo el historial de chat y solicite al LLM un resultado final.

In [25]:
consulta = "¿Cuánto debo dar de propina por $60 al 20%?"

llm_con_herramientas_2 = llm.bind_tools([calcular_propina])
historial_chat_2 = [HumanMessage(content= consulta)]

respuesta = llm_con_herramientas_2.invoke(historial_chat_2)

llamar_herramienta = respuesta.tool_calls[0]
nombre_herramienta = llamar_herramienta["name"]
args_herramienta = llamar_herramienta["args"]
call_id_herramienta = llamar_herramienta["id"]

resultado_herramienta = map_herramientas_2[nombre_herramienta].invoke(args_herramienta)
herramienta_message = ToolMessage(content= str(resultado_herramienta), tool_call_id= call_id_herramienta)

historial_chat_2.extend([respuesta, herramienta_message])

respuesta_final = llm_con_herramientas_2.invoke(historial_chat_2)
print(respuesta_final.content)

Debes dar una propina de $12 por una factura de $60 al 20%.


### *Ejercicio 3: Crear un Agente para Calcular Propinas*

Cree un agente para automatizar todo el proceso que completó anteriormente.

In [26]:
class AgenteCalculoPropina:
    def __init__(self, llm):
        self.llm_con_herramientas = llm.bind_tools([calcular_propina])
        self.map_herramientas = {"calcular_propina": calcular_propina}

    def run(self, consulta: str) -> str:
        historial_chat = [HumanMessage(content= consulta)]
        respuesta = self.llm_con_herramientas.invoke(historial_chat)

        if not respuesta.tool_calls:
            return respuesta.content
        
        llamar_herramienta = respuesta.tool_calls[0]
        nombre_herramienta = llamar_herramienta["name"]
        args_herramienta = llamar_herramienta["args"]
        call_id_herramienta = llamar_herramienta["id"]

        resultado_herramienta = self.map_herramientas[nombre_herramienta].invoke(args_herramienta)
        herramienta_message = ToolMessage(content= str(resultado_herramienta), tool_call_id= call_id_herramienta)
        historial_chat.extend([respuesta, herramienta_message])

        respuesta_final = self.llm_con_herramientas.invoke(historial_chat)
        
        return respuesta_final.content

consulta = "¿Cuánto debo dar de propina por $60 al 20%?"

agente_propina = AgenteCalculoPropina(llm)

print(agente_propina.run(consulta))

Debes dar una propina de $12 por una factura de $60 al 20%.


Copyright © IBM Corporation. All rights reserved.


<!-- ## Changelog

| Date | Version | Changed by | Change Description |
|------|--------|--------|---------|
| 2024-06-06 | 0.1 |  P. Kravitz | ID review and edit. No code edits.Updated the copyright statement. Change log added. Instructional edits only for IBM style. Second person, accessibility, and other minor grammar edits.| -->
